# Attaching a fragment made of many residues

The fragment is the 31-residue semaglutide peptide, taken as a `Chain`
whose children are bonded `Residue` objects. It is attached through the
alpha carbon of its last glycine to lysine 48 of ubiquitin. Every fragment
residue keeps its name, its atoms and its formal charge.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
from mbuild.biopolymers import Protein

fragment = Protein("../semaglutide_apo.pdb").chains[0]
print(len(fragment.children), "residues:", [(r.name, r.resnum) for r in fragment.children][:5], "...")

protein = Protein("../1ubq_protonated.pdb")
protein.deprotonate(48, "NZ")
bond = protein.attach(fragment, fragment_atom_name="CA", fragment_resnum=31, resnum=48, atom_name="NZ", relax=False)

31 residues: [('HIS', 1), ('AIB', 2), ('GLU', 3), ('GLY', 4), ('THR', 5)] ...


In [3]:
residues = list(protein.residues())
print(len(residues), "residues,", protein.n_particles, "atoms, net charge", protein.net_formal_charge)
print("ubiquitin ends, fragment begins:", [(r.name, r.resnum, r.formal_charge) for r in residues[74:80]])
print("fragment ends:", [(r.name, r.resnum, r.formal_charge) for r in residues[-3:]])

107 residues, 1698 atoms, net charge -2
ubiquitin ends, fragment begins: [('GLY', 75, 0), ('GLY', 76, -1), ('HIS', 77, 1), ('AIB', 78, 0), ('GLU', 79, -1), ('GLY', 80, 0)]
fragment ends: [('GLY', 105, 0), ('ARG', 106, 1), ('GLY', 107, -1)]


One bond record describes the modification. It is what a downstream residue library needs.

In [4]:
record, = protein.bond_records()
record

{'residue_names': ('LYS', 'GLY'),
 'residue_numbers': (48, 107),
 'chain_ids': ('A', 'A'),
 'icodes': ('', ''),
 'atom_names': ('NZ', 'CA'),
 'leaving_atoms': (['HZ1'], ['HA2']),
 'bond_order': 1}

In [5]:
from pathlib import Path

written = Path("../assets_cache/1ubq_plus_peptide.pdb")
protein.save_pdb(written, overwrite=True)
lines = written.read_text().splitlines()
print(sorted({(line[17:20], int(line[22:26])) for line in lines if line.startswith(("ATOM", "HETATM"))}, key=lambda t: t[1])[74:82])

[('GLY', 75), ('GLY', 76), ('HIS', 77), ('AIB', 78), ('GLU', 79), ('GLY', 80), ('THR', 81), ('PHE', 82)]


Pablo reads the file once the bond record is handed over as a crosslink.

In [6]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

library = STD_CCD_CACHE.with_crosslink(
    residues=list(record["residue_names"]),
    linking_atoms=list(record["atom_names"]),
    leaving_atoms=[list(side) for side in record["leaving_atoms"]],
    bond_order=record["bond_order"],
)
topology = topology_from_pdb(written, residue_library=library)
print(topology.n_molecules, "molecule,", topology.n_atoms, "atoms,", topology.n_bonds, "bonds, net charge", topology.molecule(0).total_charge)
assert topology.n_bonds == protein.n_bonds

1 molecule, 1698 atoms, 1710 bonds, net charge -2.0 elementary_charge
